# Stage 9 — Pilot, then prove it reproduces

**The question:** does the new pipeline work, and does inference reproduce
training exactly?

This notebook does the first half, the pilot:

1. Picks 2,000 slides, stratified on ISUP grade and hospital.
2. Extracts two feature sets over the same slides, in the per-slide format the
   full run will use:
   - **tissue**: tiles chosen by `get_tiles`, the Stage 6 selector;
   - **grid**: tiles on an evenly spaced grid that ignores tissue, a
     deliberately poor floor.
3. Checks both sets are complete and writes a manifest for each.
4. Trains two simple models on each set with the Stage 4 folds and compares the
   scores, on mixed folds and from Radboud to Karolinska.

The reproduction test (re-tile and re-encode five slides from the `.tiff` and
compare against what is stored) comes after.

Extraction skips every slide whose file already exists, so a re-run, or a run
resumed after a crash, only does the slides that are missing.

## Preamble

In [ ]:
"""Make the project's src/ folder importable.

Locally it sits one folder up from notebooks/. On Kaggle it arrives as an
attached dataset, so every attached dataset is checked for a src/ folder, and
the notebook stops with a clear message if there is none.
"""

import glob
import os
import sys

SRC_ROOT = next(
    (
        root
        for root in [os.path.abspath("..")] + sorted(glob.glob("/kaggle/input/*"))
        if os.path.isfile(os.path.join(root, "src", "tiling.py"))
    ),
    None,
)
if SRC_ROOT is None:
    raise FileNotFoundError("src/ not found; on Kaggle, attach the project's src folder as a dataset")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

In [ ]:
"""Imports.

datetime, json, platform, subprocess  record the manifest: date, settings, code commit.
hashlib    fingerprints the encoder weights for the manifest.
numpy, pandas, torch, timm, tqdm      the usual.
sklearn    picks the pilot, scores with quadratic weighted kappa, and is the linear head.
src.paths  where the data, derived files and weights are.
src.encode and src.tiling  the encoder and the tile selector, and their settings for the manifest.
"""

import datetime
import hashlib
import json
import os
import platform
import subprocess

import numpy as np
import pandas as pd
import timm
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

from src import encode, tiling
from src.encode import DEVICE, FEATURE_DIM, encode_tiles, load_encoder_offline, prepare
from src.paths import DATA_DIR, DERIVED_DIR, find_input, slide_path, weights_path
from src.tiling import get_tiles

In [ ]:
"""Where the feature sets go, and what they are called.

1. FEATURES_DIR holds one folder per feature set, named by its version string
   (handbook Part 7). It sits next to derived/: data/features/ locally, and
   /kaggle/working/features/ on Kaggle, because /kaggle/input is read-only.
2. A version string names what produced a set: encoder, read level, tile size,
   tile cap, how the tiles were placed, and a number that goes up whenever any
   of those change.
   - TISSUE_VERSION: tiles from get_tiles, placed by the tissue detector.
   - GRID_VERSION: tiles on an evenly spaced grid, tissue ignored.
3. Each set's folder holds slides/<slide_id>.pt, one file per slide, and
   manifest.json.
"""

FEATURES_DIR = os.path.join(os.path.dirname(os.path.normpath(DERIVED_DIR)), "features")
TISSUE_VERSION = "uni2h-L1-224-t144-hsv-v1"
GRID_VERSION = "uni2h-L1-224-g144-grid-v1"

In [ ]:
"""Seeds and sizes.

PILOT_SEED picks the pilot slides. It is 42 rather than the project's usual 0
because the pilot was first drawn with 42; changing it would pick different
slides.
SEED drives everything else: model initialisation, shuffling, the bootstrap.
"""

PILOT_SEED = 42
PILOT_SIZE = 2000
SEED = 0

## Code

In [ ]:
df = pd.read_parquet(find_input("slide_inventory.parquet"))
df.head()

In [ ]:
df["unified_label"] = list(zip(df["isup_grade"],df["provider"]))
df = df[["slide_id","unified_label"]]
df.head()

In [ ]:
selected_stratified_portion , _ = train_test_split(
    df,
    train_size = PILOT_SIZE,
    stratify = df["unified_label"],
    random_state = PILOT_SEED
)
pilot_ids = selected_stratified_portion["slide_id"].tolist()
selected_stratified_portion

In [ ]:
encoder = load_encoder_offline(weights_path())
encoder = prepare(encoder)

## The per-slide format

One file per slide, `slides/<slide_id>.pt`, holding:

| Key | What |
|---|---|
| `version` | the feature set's version string |
| `slide_id` | the slide |
| `features` | float16 `[n, 1536]`, from `encode_tiles` |
| `coords` | int32 `[n, 2]`, level-0 (x, y) of each tile's top-left corner, in the order of `features` |
| `tissue_frac` | float32 `[n]` for the tissue set; `None` for the grid set, which never measures tissue |
| `gpu` | the device that encoded it: features are repeatable on one GPU but differ slightly between GPUs (Stage 7) |

Everything is a tensor, a string or `None`, so it loads with
`torch.load(..., weights_only=True)`.

In [ ]:
"""The per-slide format, and the loop that fills it.

save_slide writes one slide's file. It writes to a temporary name and renames
it at the end, so a crash mid-write never leaves a file that looks finished,
which the skip below would then trust.

extract_set runs one tiling function over the slides:
1. Make the set's slides/ folder.
2. Skip every slide whose file already exists. A re-run, or a run resumed after
   a crash, only does the slides that are missing.
3. For each remaining slide: tile, encode, save. A slide with no tiles is saved
   with n = 0 rather than skipped, so load_set reports it instead of it
   quietly vanishing.

tile_fn takes a slide path and returns (tiles, coords, tissue_frac), with
tissue_frac None when the tiling does not measure it.
"""

GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"


def slide_file(version, slide_id):
    return os.path.join(FEATURES_DIR, version, "slides", f"{slide_id}.pt")


def save_slide(version, slide_id, features, coords, tissue_frac):
    record = {
        "version": version,
        "slide_id": slide_id,
        "features": features,
        "coords": torch.from_numpy(np.asarray(coords, dtype=np.int32).reshape(-1, 2)),
        "tissue_frac": None if tissue_frac is None else torch.from_numpy(np.asarray(tissue_frac, dtype=np.float32)),
        "gpu": GPU_NAME,
    }
    path = slide_file(version, slide_id)
    torch.save(record, path + ".part")
    os.replace(path + ".part", path)


def extract_set(version, tile_fn, slide_ids, encoder):
    os.makedirs(os.path.join(FEATURES_DIR, version, "slides"), exist_ok=True)
    todo = [s for s in slide_ids if not os.path.exists(slide_file(version, s))]
    print(f"{version}: {len(slide_ids) - len(todo)} already done, {len(todo)} to do")
    for slide_id in tqdm(todo, desc=version):
        tiles, coords, tissue_frac = tile_fn(slide_path(slide_id))
        save_slide(version, slide_id, encode_tiles(encoder, tiles), coords, tissue_frac)

In [ ]:
extract_set(TISSUE_VERSION, get_tiles, pilot_ids, encoder)

## Exhaustive grid: the floor to beat

The second pilot set uses a plain evenly spaced grid over the whole slide, and ignores tissue entirely. Most of its tiles land on blank glass, which is the point: it is a deliberately poor way to choose tiles. If choosing tiles by tissue does not clearly beat it, the tissue step is not doing its job.

Only the tile *positions* differ from the tissue set. The read level, the tile size, the reading (`read_tiles`), the encoder and the transform (`encode_tiles`) are the same, so any difference in score comes from where the tiles were placed.

In [ ]:
import math

import openslide

from src.tiling import READ_LEVEL, TILE_PER_SLIDE_CAP, TILE_SIZE, read_tiles


def get_grid_tiles(slide_path, n=TILE_PER_SLIDE_CAP):
    """Up to n tiles on an evenly spaced grid over the whole slide, tissue or not.

    1. Cut the read level into whole TILE_SIZE cells: n_cols across, n_rows down.
    2. Split n between columns and rows in proportion to the slide's shape, so
       the spacing is about the same both ways. A long, narrow slide gets few
       columns and many rows. Neither can exceed the cells that exist.
    3. Spread the chosen columns and rows evenly from edge to edge (linspace),
       and take every (column, row) pair, in reading order.
    4. Convert cells to level-0 (x, y): a cell is TILE_SIZE pixels at READ_LEVEL,
       which is TILE_SIZE x that level's downsample in level-0 pixels.
    5. Read the tiles with the same read_tiles as the tissue set, and close the
       slide in a finally block.

    Returns tiles and coords, like get_tiles. There is no randomness, so a slide
    always gives the same tiles.
    """
    slide = openslide.OpenSlide(slide_path)
    try:
        width, height = slide.level_dimensions[READ_LEVEL]
        n_cols, n_rows = width // TILE_SIZE, height // TILE_SIZE

        k_cols = min(max(1, round(math.sqrt(n * n_cols / n_rows))), n_cols)
        k_rows = min(n // k_cols, n_rows)

        cols = np.unique(np.linspace(0, n_cols - 1, k_cols).round().astype(int))
        rows = np.unique(np.linspace(0, n_rows - 1, k_rows).round().astype(int))
        cells = np.array([(col, row) for row in rows for col in cols])

        coords = np.round(cells * TILE_SIZE * slide.level_downsamples[READ_LEVEL]).astype(np.int32)
        tiles = read_tiles(slide, coords, READ_LEVEL)
    finally:
        slide.close()
    return tiles, coords

In [ ]:
"""The grid set. get_grid_tiles returns no tissue fraction, because the grid
never looks at tissue, so it is stored as None."""


def grid_tiles(slide_path):
    tiles, coords = get_grid_tiles(slide_path)
    return tiles, coords, None


extract_set(GRID_VERSION, grid_tiles, pilot_ids, encoder)

## Are both sets complete?

Nothing downstream should trust a feature set nobody checked. `load_set` reads
every file of a set and checks it, and returns the features for the models
below. `write_manifest` then records what the set is, including the explicit
flag that label masks were not used.

In [ ]:
"""Load and check a feature set, then write its manifest.

load_set(version, slide_ids, has_tissue_frac):
1. The slides/ folder must hold exactly these slides: none missing, none extra,
   no leftover .part file from a crashed write.
2. Every file must carry the right version and slide_id; features float16
   [n, FEATURE_DIM], all finite; coords int32 [n, 2]; tissue_frac float32 [n]
   for the tissue set and None for the grid set.
3. Returns the features per slide, the tile count per slide, and the GPUs that
   encoded them. A slide with zero tiles is listed, not failed: it is a finding,
   and the models below leave it out of both sets.

write_manifest(version, placement, counts, gpus) writes manifest.json with the
fields handbook Part 7 asks for. The weights fingerprint identifies the encoder
exactly; Stage 7 recorded the expected value.
"""


def load_set(version, slide_ids, has_tissue_frac):
    folder = os.path.join(FEATURES_DIR, version, "slides")
    on_disk = set(os.listdir(folder))
    leftovers = [f for f in on_disk if not f.endswith(".pt")]
    on_disk = {f[:-3] for f in on_disk if f.endswith(".pt")}
    missing, extra = set(slide_ids) - on_disk, on_disk - set(slide_ids)
    assert not missing and not extra and not leftovers, (
        f"{version}: {len(missing)} missing, {len(extra)} extra, {len(leftovers)} other files"
    )

    bags, counts, gpus = {}, {}, set()
    for slide_id in tqdm(slide_ids, desc=f"checking {version}"):
        record = torch.load(slide_file(version, slide_id), weights_only=True)
        features = record["features"]
        n = features.shape[0]
        assert record["version"] == version and record["slide_id"] == slide_id, slide_id
        assert features.dtype == torch.float16 and features.shape == (n, FEATURE_DIM), slide_id
        assert torch.isfinite(features).all(), slide_id
        assert record["coords"].dtype == torch.int32 and record["coords"].shape == (n, 2), slide_id
        if has_tissue_frac:
            assert record["tissue_frac"].dtype == torch.float32 and record["tissue_frac"].shape == (n,), slide_id
        else:
            assert record["tissue_frac"] is None, slide_id
        bags[slide_id], counts[slide_id] = features, n
        gpus.add(record["gpu"])

    counts = pd.Series(counts)
    print(f"{version}: {len(counts)} slides, {counts.sum():,} tiles, per slide median {counts.median():.0f}, "
          f"min {counts.min()}, max {counts.max()}; encoded on {sorted(gpus)}")
    print(f"slides with zero tiles: {sorted(counts[counts == 0].index)}")
    return bags, counts, gpus


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            digest.update(block)
    return digest.hexdigest()


def code_commit():
    try:
        return subprocess.run(["git", "describe", "--always", "--dirty"], capture_output=True, text=True,
                              check=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        return "unknown"


def write_manifest(version, placement, counts, gpus):
    manifest = {
        "version": version,
        "encoder": encode.HF_MODEL,
        "architecture": encode.ARCHITECTURE,
        "weights_sha256": WEIGHTS_SHA256,
        "feature_dim": FEATURE_DIM,
        "dtype": "float16",
        "encoded_on": sorted(gpus),
        "tile_size": tiling.TILE_SIZE,
        "read_level": tiling.READ_LEVEL,
        "tile_cap": tiling.TILE_PER_SLIDE_CAP,
        "placement": placement,
        "masks_used": False,
        "coordinate_space": "level-0 (x, y) of each tile's top-left corner",
        "slides": len(counts),
        "tiles": int(counts.sum()),
        "zero_tile_slides": sorted(counts[counts == 0].index),
        "slide_selection": {
            "source": "slide_inventory.parquet",
            "method": "sklearn train_test_split, stratified on (isup_grade, provider)",
            "train_size": PILOT_SIZE,
            "random_state": PILOT_SEED,
        },
        "torch": torch.__version__,
        "timm": timm.__version__,
        "python": platform.python_version(),
        "created": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
        "code_commit": code_commit(),
    }
    path = os.path.join(FEATURES_DIR, version, "manifest.json")
    with open(path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"wrote {path}")


WEIGHTS_SHA256 = file_sha256(weights_path())
print(f"weights sha256: {WEIGHTS_SHA256}")

In [ ]:
"""Check both sets and write their manifests.

The tissue set's placement records every tiling setting from src/tiling.py;
the grid set's records that it ignores tissue.
"""

tissue_bags, tissue_counts, tissue_gpus = load_set(TISSUE_VERSION, pilot_ids, has_tissue_frac=True)
write_manifest(TISSUE_VERSION, {
    "method": "src.tiling.get_tiles: tissue mask on the smallest level, grid of tile-sized cells, "
              "best cells by tissue fraction, then reading order",
    "saturation_cut": tiling.SATURATION_CUT,
    "min_cc_area": tiling.MIN_CC_AREA,
    "ink_hue": list(tiling.INK_HUE_THRESH),
    "ink_saturation": tiling.INK_SAT_THRESH,
    "min_tissue_frac": tiling.MIN_FRAC,
    "pad_edges": tiling.PAD_EDGES,
}, tissue_counts, tissue_gpus)

grid_bags, grid_counts, grid_gpus = load_set(GRID_VERSION, pilot_ids, has_tissue_frac=False)
write_manifest(GRID_VERSION, {
    "method": "get_grid_tiles in notebooks/09_pilot_study.ipynb: evenly spaced grid over the whole slide at "
              "the read level, tissue ignored; a floor, not a candidate for the full run",
    "max_tiles": tiling.TILE_PER_SLIDE_CAP,
}, grid_counts, grid_gpus)

## Does tissue beat the grid?

The same two models are trained on each feature set, on the same slides, with
the same folds. Only the tile placement differs, so a gap in score is the tissue
step's contribution.

**Two heads.** Both read bags of tile features; neither ever sees an image.
- *Mean pool + logistic regression*: average a slide's tile features,
  standardise, and fit a 6-class logistic regression with scikit-learn's
  defaults. Nothing to tune. Blank-glass tiles dilute the average, so this head
  is the harsher test for the grid.
- *Gated ABMIL* (Ilse et al., 2018): learns a weight for every tile and averages
  with those weights, so it can learn to ignore glass. Fixed settings in the
  style of the published PANDA benchmarks: hidden 512, attention 256, dropout
  0.25, AdamW with learning rate 1e-4 and weight decay 1e-5, cosine schedule,
  20 epochs, one slide per step, 6-class cross-entropy, grade = argmax. No
  early stopping: stopping on the scored fold would leak, and there is no
  separate validation fold in a pilot this size. Nothing is tuned.

**Two protocols**, both from `data/folds.csv`:
- *Mixed*: for each fold, train on the other four and predict it. The
  predictions for all pilot slides are pooled and scored once, and also scored
  per hospital.
- *Radboud to Karolinska*: for each fold, train on the Radboud slides of the
  other four and predict the Karolinska slides of this one. Pooled, that scores
  every pilot Karolinska slide by a model that never saw Karolinska.

**Is the gap real?** Both sets are scored on the same slides, so the
difference is bootstrapped by resampling slides: 2,000 resamples, and the 95%
interval of tissue minus grid.

Labels come from the fold file and are read here for the first time in this
notebook. Nothing above this line reads them.

In [ ]:
"""Labels and folds for the pilot slides, from the Stage 4 fold file.

Slides with zero tiles in either set are left out of both, so the two sets
are always scored on exactly the same slides.
"""

folds = pd.read_csv(find_input("folds.csv", local_dir=DATA_DIR)).set_index("slide_id")
scored_ids = [s for s in pilot_ids if tissue_counts[s] > 0 and grid_counts[s] > 0]
labels = folds.loc[scored_ids]
print(f"scored slides: {len(scored_ids)} of {len(pilot_ids)}")
print(pd.crosstab(labels["fold"], labels["provider"]).to_string())

In [ ]:
"""The two heads and the two protocols.

fit_predict_mean(bags, train_ids, test_ids): average each slide's features in
float32, then standardise and fit a logistic regression on the training
slides. Returns the predicted grade for each test slide.

GatedABMIL: each tile goes through a small layer (1536 -> 512); a gated
attention block (Ilse et al. 2018) scores every tile; the softmax of the scores
weights the tiles into one slide vector; a linear layer gives 6 grade logits.

fit_predict_abmil(bags, train_ids, test_ids):
1. Seed the model and the shuffling with SEED, so every run starts the same.
2. Train for ABMIL_EPOCHS epochs, one slide per step, in a shuffled order,
   with cross-entropy against the grade and a cosine learning rate.
3. Predict each test slide in eval mode: the argmax of its logits.

run(bags, head, protocol) loops over the five folds, picks the training and
test slides the protocol asks for, and returns the pooled predictions.
"""

ABMIL_EPOCHS = 20


def fit_predict_mean(bags, train_ids, test_ids):
    def pooled(ids):
        return torch.stack([bags[s].float().mean(dim=0) for s in ids]).numpy()

    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    model.fit(pooled(train_ids), labels.loc[train_ids, "isup_grade"].to_numpy())
    return model.predict(pooled(test_ids))


class GatedABMIL(torch.nn.Module):
    def __init__(self, in_dim=FEATURE_DIM, hidden=512, attention=256, classes=6, dropout=0.25):
        super().__init__()
        self.embed = torch.nn.Sequential(torch.nn.Linear(in_dim, hidden), torch.nn.ReLU(), torch.nn.Dropout(dropout))
        self.attn_v = torch.nn.Sequential(torch.nn.Linear(hidden, attention), torch.nn.Tanh(), torch.nn.Dropout(dropout))
        self.attn_u = torch.nn.Sequential(torch.nn.Linear(hidden, attention), torch.nn.Sigmoid(),
                                          torch.nn.Dropout(dropout))
        self.attn_w = torch.nn.Linear(attention, 1)
        self.classify = torch.nn.Linear(hidden, classes)

    def forward(self, tiles):
        h = self.embed(tiles)
        weights = torch.softmax(self.attn_w(self.attn_v(h) * self.attn_u(h)).squeeze(-1), dim=0)
        return self.classify(weights @ h)


def fit_predict_abmil(bags, train_ids, test_ids):
    torch.manual_seed(SEED)
    order = torch.Generator().manual_seed(SEED)
    model = GatedABMIL().to(DEVICE)
    optimiser = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=ABMIL_EPOCHS * len(train_ids))
    grades = torch.tensor(labels.loc[train_ids, "isup_grade"].to_numpy(), device=DEVICE)

    model.train()
    for _ in range(ABMIL_EPOCHS):
        for i in torch.randperm(len(train_ids), generator=order).tolist():
            logits = model(bags[train_ids[i]].to(DEVICE).float())
            loss = torch.nn.functional.cross_entropy(logits[None], grades[i:i + 1])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            schedule.step()

    model.eval()
    with torch.inference_mode():
        return np.array([model(bags[s].to(DEVICE).float()).argmax().item() for s in test_ids])


def run(bags, head, protocol):
    predictions = {}
    for fold in range(5):
        train = labels["fold"] != fold
        test = labels["fold"] == fold
        if protocol == "radboud_to_karolinska":
            train &= labels["provider"] == "radboud"
            test &= labels["provider"] == "karolinska"
        train_ids, test_ids = labels.index[train].tolist(), labels.index[test].tolist()
        predictions.update(zip(test_ids, head(bags, train_ids, test_ids)))
    return pd.Series(predictions)

In [ ]:
"""Train and predict: 2 feature sets x 2 heads x 2 protocols, five folds each.

The ABMIL runs take about a minute per fold on the RTX 4060.
"""

feature_sets = {"tissue": tissue_bags, "grid": grid_bags}
heads = {"mean_pool": fit_predict_mean, "abmil": fit_predict_abmil}
protocols = ["mixed", "radboud_to_karolinska"]

predictions = {}
for (set_name, bags), (head_name, head), protocol in tqdm(
    [(s, h, p) for s in feature_sets.items() for h in heads.items() for p in protocols], desc="runs"
):
    predictions[set_name, head_name, protocol] = run(bags, head, protocol)

In [ ]:
"""Score every run, and bootstrap the tissue-minus-grid gap.

1. qwk: quadratic weighted kappa, with grades 0-5 as the fixed label set, so a
   grade nobody predicted still counts.
2. For each head and protocol: score both sets on the same slides; for the
   mixed protocol also score each hospital separately.
3. Bootstrap: draw the scored slides with replacement 2,000 times (seeded),
   score both sets on each draw, and take the 2.5th and 97.5th percentiles of
   the difference. An interval that stays above 0 means tissue wins on these
   slides beyond resampling noise.
"""

GRADES = list(range(6))
BOOTSTRAP_DRAWS = 2000


def qwk(truth, predicted):
    return cohen_kappa_score(truth, predicted, labels=GRADES, weights="quadratic")


rng = np.random.default_rng(SEED)
rows = []
for head_name in heads:
    for protocol in protocols:
        ids = predictions["tissue", head_name, protocol].index
        groups = {"all": ids}
        if protocol == "mixed":
            groups.update({p: ids[labels.loc[ids, "provider"] == p] for p in ("karolinska", "radboud")})
        for group, group_ids in groups.items():
            truth = labels.loc[group_ids, "isup_grade"].to_numpy()
            tissue = predictions["tissue", head_name, protocol].loc[group_ids].to_numpy()
            grid = predictions["grid", head_name, protocol].loc[group_ids].to_numpy()
            draws = []
            for _ in range(BOOTSTRAP_DRAWS):
                sample = rng.integers(0, len(truth), len(truth))
                draws.append(qwk(truth[sample], tissue[sample]) - qwk(truth[sample], grid[sample]))
            low, high = np.percentile(draws, [2.5, 97.5])
            rows.append({
                "head": head_name, "protocol": protocol, "slides": group, "n": len(truth),
                "tissue": qwk(truth, tissue), "grid": qwk(truth, grid),
                "difference": qwk(truth, tissue) - qwk(truth, grid), "95% low": low, "95% high": high,
            })

results = pd.DataFrame(rows)
print(f"feature sets: tissue = {TISSUE_VERSION}, grid = {GRID_VERSION}\n")
print(results.round(3).to_string(index=False))

## Result

To be filled in after the run: the table above, which versions produced it,
and whether tissue clearly beats the grid under both heads and both protocols.